# Text Generation

This notebook implements autoregressive text generation from a GPT model with
support for **greedy decoding**, **top-k sampling**, **top-p (nucleus) sampling**,
and **temperature scaling**.

### Key Concepts

- **Greedy decoding** (`temperature=0`): always pick the highest-probability token.
- **Temperature**: scales logits before softmax — lower values sharpen the distribution.
- **Top-k**: keep only the *k* most probable tokens, zero out the rest.
- **Top-p (nucleus)**: keep the smallest set of tokens whose cumulative probability exceeds *p*.
- Generation stops at `<eos>` or when `max_new_tokens` is reached.

---
## Imports and Setup

In [ ]:
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

---
## TextGenerator Class

The `TextGenerator` wraps a GPT model and a BPE tokenizer. It encodes the
prompt, runs autoregressive generation with the chosen sampling strategy,
and decodes the result back to text.

The `_sample_token` static method implements all four strategies in a single
code path:
1. If `temperature == 0` → greedy (argmax).
2. Otherwise, scale logits by temperature.
3. Optionally apply top-k filtering.
4. Optionally apply top-p (nucleus) filtering.
5. Sample from the resulting distribution.

In [ ]:
class TextGenerator:
    """Autoregressive text generator for GPT models."""

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.eos_token_id = tokenizer.SPECIAL_TOKENS['<eos>']

    @torch.no_grad()
    def generate(self, prompt, max_new_tokens=200, temperature=1.0,
                 top_k=None, top_p=None):
        """Generate text from a prompt string.

        Returns (generated_text, generated_token_ids)."""
        if not prompt:
            raise ValueError('Prompt must be a non-empty string.')
        self.model.eval()
        prompt_ids = self.tokenizer.encode(prompt)
        device = next(self.model.parameters()).device
        input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=device)
        generated_ids = []
        max_seq_len = self.model.config.max_seq_len

        for _ in range(max_new_tokens):
            if input_ids.size(1) > max_seq_len:
                input_ids = input_ids[:, -max_seq_len:]
            logits = self.model(input_ids)
            next_logits = logits[:, -1, :]
            next_token_id = self._sample_token(
                next_logits, temperature=temperature, top_k=top_k, top_p=top_p)
            generated_ids.append(next_token_id)
            if next_token_id == self.eos_token_id:
                break
            next_tensor = torch.tensor(
                [[next_token_id]], dtype=torch.long, device=device)
            input_ids = torch.cat([input_ids, next_tensor], dim=1)

        generated_text = self.tokenizer.decode(generated_ids)
        return generated_text, generated_ids


### Token Sampling

The `_sample_token` method handles all sampling strategies:
- **Greedy**: `temperature=0` → `argmax`
- **Temperature scaling**: divide logits by temperature before softmax
- **Top-k**: zero out all but the top-k logits
- **Top-p**: zero out tokens beyond the cumulative probability threshold

In [ ]:
    @staticmethod
    def _sample_token(logits, temperature, top_k, top_p):
        """Sample a single token from logits."""
        logits = logits.squeeze(0)
        if temperature == 0:
            return logits.argmax(dim=-1).item()
        logits = logits / temperature

        # Top-k filtering
        if top_k is not None and top_k > 0:
            top_k = min(top_k, logits.size(-1))
            kth_val = logits.topk(top_k).values[-1]
            logits = logits.where(
                logits >= kth_val,
                torch.tensor(float('-inf'), device=logits.device))

        # Top-p (nucleus) filtering
        if top_p is not None and 0.0 < top_p < 1.0:
            sorted_logits, sorted_indices = logits.sort(descending=True)
            cumulative_probs = F.softmax(sorted_logits, dim=-1).cumsum(dim=-1)
            sorted_mask = cumulative_probs - F.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[sorted_mask] = float('-inf')
            logits = sorted_logits.scatter(0, sorted_indices, sorted_logits)

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        return next_token.item()

# Attach the static method to the class
TextGenerator._sample_token = TextGenerator._sample_token

---
## Demo: Generation with Different Strategies

We create a tiny GPT model and a BPE tokenizer trained on a small corpus,
then demonstrate generation with each sampling strategy.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.config import GPTConfig
from src.model.transformer import GPTModel
from src.tokenizer.bpe_tokenizer import BPETokenizer

# Train a small tokenizer on sample text
sample_text = (
    'the cat sat on the mat '
    'the dog sat on the log '
    'a bird flew over the tree '
    'the sun shines bright today '
) * 20

tokenizer = BPETokenizer(vocab_size=300)
tokenizer.train(sample_text)
print(f'Tokenizer vocab size: {len(tokenizer.vocab)}')

# Create a small GPT model
config = GPTConfig(
    vocab_size=len(tokenizer.vocab),
    d_model=64,
    num_heads=4,
    num_layers=2,
    max_seq_len=128,
    dropout_rate=0.0,
)
model = GPTModel(config)
print(f'Model parameters: {model.count_parameters():,}')

### Greedy Decoding (`temperature=0`)

Greedy decoding always picks the most probable next token. It is fully
deterministic — running it twice on the same prompt produces identical output.

In [ ]:
gen = TextGenerator(model, tokenizer)

text, ids = gen.generate('the cat', max_new_tokens=20, temperature=0)
print('Greedy output:', repr(text))
print('Token IDs:', ids)

# Verify determinism
text2, ids2 = gen.generate('the cat', max_new_tokens=20, temperature=0)
assert text == text2 and ids == ids2, 'Greedy decoding should be deterministic'
print('Determinism check passed!')

### Temperature Sampling (`temperature=0.8`)

Temperature < 1 sharpens the distribution (more confident picks).
Temperature > 1 flattens it (more random). Results are stochastic.

In [ ]:
text, ids = gen.generate('the cat', max_new_tokens=20, temperature=0.8)
print('Temperature=0.8 output:', repr(text))
print('Token IDs:', ids)

### Top-k Sampling (`top_k=5`)

Only the top-k most probable tokens are considered at each step.
This prevents sampling from the long tail of unlikely tokens.

In [ ]:
text, ids = gen.generate('the cat', max_new_tokens=20, temperature=1.0, top_k=5)
print('Top-k=5 output:', repr(text))
print('Token IDs:', ids)

### Top-p (Nucleus) Sampling (`top_p=0.9`)

Keeps the smallest set of tokens whose cumulative probability exceeds *p*.
Adapts the number of candidates dynamically based on the distribution shape.

In [ ]:
text, ids = gen.generate('the cat', max_new_tokens=20, temperature=1.0, top_p=0.9)
print('Top-p=0.9 output:', repr(text))
print('Token IDs:', ids)

### Combined: Temperature + Top-k + Top-p

All strategies can be combined. Temperature is applied first, then top-k,
then top-p filtering.

In [ ]:
text, ids = gen.generate(
    'the cat', max_new_tokens=20,
    temperature=0.7, top_k=10, top_p=0.95)
print('Combined output:', repr(text))
print('Token IDs:', ids)

---
## Output Consistency Check

The generator returns `(text, token_ids)`. Decoding `token_ids` with the
tokenizer should reproduce `text`.

In [ ]:
text, ids = gen.generate('the', max_new_tokens=15, temperature=0)
decoded = tokenizer.decode(ids)
print('Generated text:', repr(text))
print('Decoded IDs:  ', repr(decoded))
assert text == decoded, f'Mismatch: {text!r} != {decoded!r}'
print('Consistency check passed!')

---
## Summary

| Strategy | Deterministic? | Key Parameter |
|---|---|---|
| Greedy | Yes | `temperature=0` |
| Temperature | No | `temperature` (0, ∞) |
| Top-k | No | `top_k` (integer) |
| Top-p | No | `top_p` (0, 1) |

All strategies can be combined. The `TextGenerator` class mirrors
`src/generation/generator.py` and is ready for use with any trained GPT checkpoint.